# Preprocessing and evaluation split

The dataset is divided chronologically into train (70%), validation (15%), and test (15%) partitions. A chronological split is preferable to a random split because delay patterns can change over time, and a model should be evaluated on later observations rather than on randomly mixed records from the same period.

The `planned_time` column is created only to order the observations: `arrival_planned_time` is preferred and `departure_planned_time` is used when the arrival schedule is unavailable. Rows with neither timestamp cannot be placed reliably in time order and are removed before splitting.

After the split, preprocessing is applied independently to each partition. This keeps the evaluation boundary explicit and prevents rows from one partition being used to build another partition's derived values. Delays below -1 minute and above 180 minutes are removed as extreme values. The first station is derived from `train_line_station_num == 1` within each `train_line_ride_id`; this identifies the actual train-line journey rather than grouping different journeys by train or line number. Rows without a first station and RE rows without a line number are removed. `time` is converted to `day_of_week`, and the two source schedule columns are replaced by the unified `planned_time` column. `train_line_station_num` is retained because later stops may have accumulated more delay. `station_name` is retained as the human-readable station feature. Technical identifiers (`eva`, `source_file`, and `train_line_ride_id`) are excluded from the processed features.

The validation partition is reserved for model selection and tuning. The test partition is kept untouched until final evaluation. Cross-validation is not used in this initial design because a random CV procedure could mix later and earlier records; if CV is needed for tuning, it should be performed chronologically within the training period.

The full dataset is large. Restart the notebook kernel before rerunning this workflow if a previous run left large DataFrames in memory.


In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from prep_job import (
    add_split_time,
    add_target,
    chronological_split,
    load_monthly_data,
    preprocess_partitions,
)

columns = [
    "arrival_planned_time",
    "departure_planned_time",
    "time",
    "delay_in_min",
    "arrival_is_canceled",
    "train_type",
    "line_number",
    "station_name",
    "train_number",
    "train_line_ride_id",
    "train_line_station_num",
    "final_destination_station",
]

data = load_monthly_data(PROJECT_ROOT / "data", columns)
labelled = add_target(data)
split_ready = add_split_time(labelled)
partitions = chronological_split(split_ready)

split_summary = pd.DataFrame(
    {
        "partition": list(partitions),
        "rows": [len(partitions[name]) for name in partitions],
    }
)
split_summary["percent"] = (
    split_summary["rows"] / len(split_ready) * 100
).round(2)
split_summary


,partition,rows,percent
0,train,10231017,70.0
1,validation,2192361,15.0
2,test,2192361,15.0


In [2]:
processed = preprocess_partitions(
    partitions,
    fill_missing_line_number=None,
    fill_missing_final_destination=False,
)

preprocessing_summary = pd.DataFrame(
    {
        "partition": list(partitions),
        "rows_before": [len(partitions[name]) for name in partitions],
        "rows_after": [len(processed[name]) for name in processed],
    }
)
preprocessing_summary["removed"] = (
    preprocessing_summary["rows_before"]
    - preprocessing_summary["rows_after"]
)
preprocessing_summary


,partition,rows_before,rows_after,removed
0,train,10231017,9864275,366742
1,validation,2192361,2120356,72005
2,test,2192361,2126112,66249


In [3]:
for name, partition in processed.items():
    print(f"{name} head")
    display(partition.head())


train head


,time,delay_in_min,arrival_is_canceled,train_type,line_number,station_name,train_number,train_line_station_num,final_destination_station,target,planned_time,day_of_week,first_station
0,2026-01-01 00:31:00,71.0,False,RE,RE30,Frankfurt (Main) Hbf,4168,1,Kassel Hbf,severe_or_cancelled,2025-12-31 23:20:00,Thursday,Frankfurt (Main) Hbf
1,2026-01-01 00:08:00,47.0,False,RE,RE70,Kiel Hbf,11236,8,NaN,severe_or_cancelled,2025-12-31 23:21:00,Thursday,Hamburg Hbf
2,2026-01-01 00:03:00,26.0,False,ICE,NaN,Mannheim Hbf,677,12,Karlsruhe Hbf,delayed,2025-12-31 23:29:00,Thursday,Berlin Ostbahnhof
4,2026-01-01 00:06:00,37.0,False,RE,RE200,Ulm Hbf,70498,1,Wendlingen(Neckar),severe_or_cancelled,2025-12-31 23:29:00,Thursday,Ulm Hbf
5,2026-01-01 00:16:00,47.0,False,RE,RE9,Osnabrück Hbf,14071,10,NaN,severe_or_cancelled,2025-12-31 23:29:00,Thursday,Bremen Hbf


validation head


,time,delay_in_min,arrival_is_canceled,train_type,line_number,station_name,train_number,train_line_station_num,final_destination_station,target,planned_time,day_of_week,first_station
10231017,2026-07-10 23:36:00,174.0,False,RE,RE42,Mönchengladbach Hbf,10158,21,NaN,severe_or_cancelled,2026-07-10 20:42:00,Friday,Münster (Westf) Hbf
10231018,2026-07-10 20:43:00,0.0,False,RE,RE32,Pegnitz,3092,4,NaN,on_time,2026-07-10 20:43:00,Friday,Bayreuth Hbf
10231019,2026-07-10 20:43:00,0.0,False,RE,RE3,Wilmersdorf (b Angermünde),3316,24,Anklam,on_time,2026-07-10 20:43:00,Friday,Lutherstadt Wittenberg Hbf
10231021,2026-07-10 20:43:00,0.0,False,RE,RE41,Haltern am See,16243,5,NaN,on_time,2026-07-10 20:43:00,Friday,Bochum Hbf
10231022,2026-07-10 20:43:00,0.0,False,ICE,NaN,Frankfurt (Main) Hbf,2589,1,Leipzig Hbf,on_time,2026-07-10 20:43:00,Friday,Frankfurt (Main) Hbf


test head


,time,delay_in_min,arrival_is_canceled,train_type,line_number,station_name,train_number,train_line_station_num,final_destination_station,target,planned_time,day_of_week,first_station
12423378,2026-08-21 17:22:00,12.0,False,ICE,NaN,Eisenach Hbf,1653,5,Dresden Hbf,delayed,2026-08-21 17:08:00,Friday,Frankfurt am Main Flughafen Fernbahnhof
12423380,2026-08-21 17:26:00,16.0,False,ICE,NaN,Erfurt Hbf,751,2,Berlin Gesundbrunnen,delayed,2026-08-21 17:08:00,Friday,Frankfurt (Main) Hbf
12423381,2026-08-21 17:27:00,1.0,False,RE,RE10b,Grombach,19280,8,Mannheim Hbf,on_time,2026-08-21 17:08:00,Friday,Bad Friedrichshall Hbf
12423382,2026-08-21 17:35:00,27.0,False,RE,RE20,Magdeburg-Eichenweiler,16943,18,Schönebeck-Bad Salzelmen,delayed,2026-08-21 17:08:00,Friday,Uelzen
12423383,2026-08-21 18:01:00,53.0,False,RE,RE5,Himmelpforten,14525,6,Cuxhaven,severe_or_cancelled,2026-08-21 17:08:00,Friday,Hamburg-Harburg
